In [6]:
import pandas as pd
import numpy as np
import re

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.metrics import accuracy_score

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\parij\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
df = pd.read_csv('../../../DataBox/IMDB_Review.csv')

# Taking Sample of 10000

df=df.sample(10000)

In [8]:
# Data PreProcessing

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Text Cleaning
def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

df['review']=df['review'].apply(clean_html)

# Convert Text To lower
def convert_lower(text):
    return text.lower()

df['review']=df['review'].apply(convert_lower)

# Remove Special character
def remove_special(text):
    x=''
    
    for i in text:
        if i.isalnum():
            x=x+i
        else:
            x=x + ' '
    return x

df['review']=df['review'].apply(remove_special)

# Remove Stop Words
def remove_stopwords(text):
    x=[]
    for i in text.split():
        
        if i not in stopwords.words('english'):
            x.append(i)
    y=x[:]
    x.clear()
    return y

df['review']=df['review'].apply(remove_stopwords)


# Convert into StemWords -> Root Form of Words
ps = PorterStemmer()

y=[]
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z=y[:]
    y.clear()
    return z

df['review']=df['review'].apply(stem_words)

# Join Back
def join_back(list_input):
    return " ".join(list_input)

df['review']=df['review'].apply(join_back)

In [ ]:
# Split I/O
X=df.iloc[:,0:1].values
y=df.iloc[:,-1].values 

# Apply Vectorization
# Bag Of Words
cv=CountVectorizer(max_features=2500)
X=cv.fit_transform(df['review']).toarray() 
 
# TTS
X_train, X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [ ]:
# Apply Various Distribution
clf1=GaussianNB()
clf2=MultinomialNB()
clf3=BernoulliNB()

# Train
clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

# Predict
y_pred1=clf1.predict(X_test)
y_pred2=clf2.predict(X_test)
y_pred3=clf3.predict(X_test)

# Check Score On Different Assumption
print("Gaussian",accuracy_score(y_test,y_pred1))
print("Multinomial",accuracy_score(y_test,y_pred2))
print("Bernaulli",accuracy_score(y_test,y_pred3))

# Observation -> Better Result with Bernaulli -> So Our data is Following Bernaulli Distribution -> 0.842

Gaussian 0.7275
Multinomial 0.8345
Bernaulli 0.842
